In this first step data are imported and visualized.

# Heading 1
## Heading 2
### Heading 3
*Italic* or _Italic_

**Bold** or __Bold__

~~Strikethrough~~


In [39]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from scipy import stats

# Load dataset
df = pd.read_csv("MyDatasetToClean.csv")
# print(df)
print(df.iloc[1]) #Print second line 
#print(df.columns)  #Getting columns


ID                                                          1
SellDate                                           2024-03-02
PayMethod                                                loan
PriceSold                                               18000
featuresIncluded      ['morepower', 'coating', 'turbocharge']
FinishPayDate                                      2088-06-24
BuyerAge                                                 23.0
CustomerFeedback              I'll recommend it to my friends
ProductColor                                           Iellow
ProductCategory                               Home Appliances
BuyerGender                                              Male
BuyerSalary                                               NaN
MonthlyInstallment                                        NaN
Name: 1, dtype: object


In [24]:
# dfmissing = df.isnull().sum()   # Number of missing values per column
# for v in df.columns:
#     print(df[v].isnull())    # Rows with missing values in 'col'
print(len(df))
df['BuyerAge'].fillna(df['BuyerAge'].mean(), inplace=True)
print(df['BuyerAge'])


20
0     45.0
1     23.0
2     67.0
3     32.0
4     44.0
5     42.0
6     38.0
7     22.0
8      4.0
9     39.0
10    40.0
11    49.0
12    68.0
13    43.0
14    39.0
15    43.0
16    30.0
17    35.0
18    35.0
19    42.0
Name: BuyerAge, dtype: float64


C:\Users\39346\AppData\Local\Temp\ipykernel_4180\3797626666.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['BuyerAge'].fillna(df['BuyerAge'].mean(), inplace=True)


Removing duplicates

In [ ]:
# Drop duplicates
# Ignore the 'ID' column when checking for duplicates
dup_rows = df[df.duplicated(subset=df.columns.difference(['ID']))]

print(dup_rows)

    ID    SellDate PayMethod  PriceSold              featuresIncluded  \
18  17         NaN      card      16000  ['morepower', 'turbocharge']   
19   5  05-05-2024      cash       8000                           NaN   

   FinishPayDate  BuyerAge      CustomerFeedback ProductColor ProductCategory  \
18    2062-07-16      35.0   This product is ok.      greeen,       Furniture   
19    2040-08-10      42.0  I would buy it again   Is Orange!     Electronics   

   BuyerGender  BuyerSalary  MonthlyInstallment  
18       Other      70093.0                 NaN  
19      Female     105813.0                 NaN  


Fixing incosistent label

In [51]:
import emoji
import re
from rapidfuzz import process

# Function to clean a string
def remove_special_characters(text):
    return re.sub(r'[^A-Za-z0-9\s]', '', str(text))

def remove_stopwords(text):
    if not isinstance(text, str):
        return text
    return ' '.join(word for word in text.split() if word.lower() not in stopwords)


def remove_emoji(text):
    return emoji.replace_emoji(text, replace='')

df['ProductColor'] = df['ProductColor'].apply(remove_special_characters)
df['ProductColor'] = df['ProductColor'].astype(str).apply(remove_emoji)

stopwords = {'is', 'the', 'a', 'an', 'of', 'in'}

df['ProductColor'] = df['ProductColor'].apply(remove_stopwords)
df['ProductColor'] = df['ProductColor'].astype(str).str.strip().str.lower()




# Get unique values
unique_values = df['ProductColor'].unique()

# Choose reference labels
known_labels = ['red', 'green', 'blue', 'yellow','white','purple','orange','black','noColor']

# Function to map each value to its best match
def correct_label(val):
    match, score, _ = process.extractOne(val, known_labels)
    return match if score > 75 else val  # Only replace if confidence is high

df['ProductColor'] = df['ProductColor'].apply(correct_label)

#If the color is not present, fill it with "NoColor"
df['ProductColor'] = df['ProductColor'].fillna('noColor')

print(df['ProductColor'])




#Print unique values
print(df['ProductColor'].unique())

0      9
1      1
2     11
3      6
4      0
5      2
6      7
7      4
8      4
9      5
10    11
11     3
12    11
13    10
14     9
15     7
16     1
17     8
18     8
19     2
Name: ProductColor, dtype: object
['9' '1' '11' '6' '0' '2' '7' '4' '5' '3' '10' '8']


Correcting values. Converting strings to dates or values.

In [41]:
import pandas as pd
import re



# Function to convert only dd-mm-yyyy formatted strings
def convert_dd_mm_yyyy(text, index=None):
    if isinstance(text, str) and re.match(r'^\d{2}-\d{2}-\d{4}$', text):
        print("This data format at index", index)
        try:
            return pd.to_datetime(text, format='%d-%m-%Y').strftime('%Y-%m-%d')
        except:
            return text
    else:
        print("Else at index", index)
        try:
            return pd.to_datetime(text, errors='coerce').strftime('%Y-%m-%d')
        except:
            return None

# Apply function with loop
for i in range(len(df)):
    df.at[i, 'SellDate'] = convert_dd_mm_yyyy(df.at[i, 'SellDate'], i)

print(df['SellDate'])


Else at index 0
Else at index 1
Else at index 2
Else at index 3
Else at index 4
Else at index 5
Else at index 6
Else at index 7
Else at index 8
Else at index 9
Else at index 10
Else at index 11
Else at index 12
Else at index 13
Else at index 14
Else at index 15
Else at index 16
Else at index 17
Else at index 18
Else at index 19
0     2025-06-13
1     2024-03-02
2     2025-12-05
3           None
4     2023-03-04
5     2024-05-05
6     2024-06-05
7     2025-02-01
8           None
9     2024-09-09
10          None
11    2023-02-18
12    2024-07-23
13    2024-03-09
14    2025-02-03
15    2025-01-05
16    2025-06-06
17          None
18          None
19    2024-05-05
Name: SellDate, dtype: object


Normalization

In [46]:
df['BuyerAge'] = (df['BuyerAge'] - df['BuyerAge'].min()) / (df['BuyerAge'].max() - df['BuyerAge'].min())
df['PriceSold'] = (df['PriceSold'] - df['PriceSold'].min()) / (df['PriceSold'].max() - df['PriceSold'].min())
df['BuyerSalary'] = (df['BuyerSalary'] - df['BuyerSalary'].min()) / (df['BuyerSalary'].max() - df['BuyerSalary'].min())

print(df['BuyerSalary'])


0     0.177919
1          NaN
2     1.000000
3     0.771260
4     0.823915
5     0.875121
6     0.876455
7     0.203460
8     0.601751
9     0.118804
10    0.266243
11    0.350086
12    0.106633
13         NaN
14    0.948210
15    0.000000
16    0.455176
17    0.502902
18    0.502902
19    0.875121
Name: BuyerSalary, dtype: float64


Label encoding of the product color column

In [52]:
df['ProductColor'] = df['ProductColor'].astype('category').cat.codes
print(df['ProductColor'])



0     11
1      1
2      3
3      8
4      0
5      4
6      9
7      6
8      6
9      7
10     3
11     5
12     3
13     2
14    11
15     9
16     1
17    10
18    10
19     4
Name: ProductColor, dtype: int8


To understand whether the customer's feedback is good or  bad I apply Sentiment Analysis, a classic Natural Language Processing (NLP) task.

In [57]:
from textblob import TextBlob


# Function to get sentiment polarity
def get_sentiment(text):
    blob = TextBlob(str(text))
    return blob.sentiment.polarity  # Ranges from -1 (negative) to 1 (positive)

# Apply to DataFrame
df['Sentiment'] = df['CustomerFeedback'].apply(get_sentiment)

# Optionally classify as positive/neutral/negative
def classify_sentiment(score):
    if score > 0.2:
        return 'Positive'
    elif score < -0.2:
        return 'Negative'
    else:
        return 'Neutral'

df['Sentiment_Label'] = df['Sentiment'].apply(classify_sentiment)

print(df['Sentiment_Label'])
# print(df['Sentiment_Label','CustomerFeedback'])

0      Neutral
1      Neutral
2     Positive
3      Neutral
4     Positive
5      Neutral
6     Positive
7      Neutral
8     Positive
9     Positive
10    Positive
11    Positive
12    Positive
13    Positive
14     Neutral
15     Neutral
16    Positive
17    Positive
18    Positive
19     Neutral
Name: Sentiment_Label, dtype: object


##Feature engineering
Compute
Monthly installment = PriceSold / Months Between SellDate and FinishPay Date
If the payment method is "loan"

- Parse "SellDate" and "FinishPayDate" into datetime objects.
- Compute the number of months between them.
- Divide PriceSold by that number to get monthly installments.

In [67]:


# from dateutil.relativedelta import relativedelta

# print(df[['FinishPayDate','SellDate']])

df['SellDate'] = pd.to_datetime(df['SellDate'], errors='coerce', dayfirst=False)
df['FinishPayDate'] = pd.to_datetime(df['FinishPayDate'], errors='coerce')

# Function to calculate months between dates
def calculate_months(start, end):
    if pd.isna(start) or pd.isna(end) or end <= start:
        return np.nan
    delta = relativedelta(end, start)
    return delta.years * 12 + delta.months + (1 if delta.days > 0 else 0)

# Add months only for loan payments
df['PaymentMonths'] = df.apply(
    lambda row: calculate_months(row['SellDate'], row['FinishPayDate']) if row['PayMethod'] == 'loan' else np.nan,
    axis=1
)

# Calculate MonthlyInstallment only for loans
df['MonthlyInstallment'] = np.where(
    df['PayMethod'] == 'loan',
    df['PriceSold'] / df['PaymentMonths'],
    np.nan
)

# Print selected columns
print(df[['PayMethod','MonthlyInstallment']])

   PayMethod  MonthlyInstallment
0       loan            0.001073
1       loan            0.000925
2       loan            0.001973
3       card                 NaN
4       cash                 NaN
5       cash                 NaN
6       loan            0.001354
7       cash                 NaN
8       card                 NaN
9       loan            0.002184
10      card                 NaN
11      card                 NaN
12      loan            0.001596
13      loan            0.001942
14      cash                 NaN
15      card                 NaN
16      loan            0.001149
17      card                 NaN
18      card                 NaN
19      cash                 NaN
